# Part 2: Simple Logistic Regression Model
We will be using a simple bag-of-words representation for our initial, simple model. To train this model, we will be using the pytorch library.

In [1]:
from modelling import BagofWordsDataset, train
from torch.utils.data import DataLoader
from pathlib import Path
import torch.nn as nn
import torch

In [2]:
DATASET_DIRECTORY = Path("data/final_dataset")
VOCABULARY = DATASET_DIRECTORY / "top10k_vocabulary.csv"
BATCH_SIZE = 64
LEARNING_RATE = 1e-4
EPOCHS = 10
L2_LAMBDA = 0.01 # add l2 reg to discourage the model from giving giant weights to extremely rare tokens

train_ds = BagofWordsDataset(DATASET_DIRECTORY / "train.csv", VOCABULARY)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE)
val_dl = DataLoader(BagofWordsDataset(DATASET_DIRECTORY / "val.csv", VOCABULARY), batch_size=BATCH_SIZE)

tokenizing 634696 documents


100%|██████████| 634696/634696 [03:00<00:00, 3511.31it/s]


tokenizing 79337 documents


100%|██████████| 79337/79337 [00:22<00:00, 3475.70it/s]


In [3]:
from modelling.logistic_regression import LogisticRegression
model = LogisticRegression(train_ds.tokenizer.size())
loss = nn.BCELoss()

train(model, loss, train_dl, val_dl, LEARNING_RATE, EPOCHS)

100%|██████████| 1240/1240 [00:01<00:00, 825.59it/s]


Epochs: 1 | Train Loss:  0.455 | Val Loss:  0.408 
[+] Saved checkpoint 'best_model_2026-03-10_13:00_epoch_0.pth'


100%|██████████| 1240/1240 [00:01<00:00, 792.47it/s]


Epochs: 2 | Train Loss:  0.392 | Val Loss:  0.389 
[+] Saved checkpoint 'best_model_2026-03-10_13:00_epoch_1.pth'


100%|██████████| 1240/1240 [00:01<00:00, 793.57it/s]


...

Epochs: 6 | Train Loss:  0.361 | Val Loss:  0.374 
[+] Saved checkpoint 'best_model_2026-03-10_13:00_epoch_5.pth'


100%|██████████| 1240/1240 [00:01<00:00, 671.25it/s]


Epochs: 7 | Train Loss:  0.359 | Val Loss:  0.374 
[+] Saved checkpoint 'best_model_2026-03-10_13:00_epoch_6.pth'


100%|██████████| 1240/1240 [00:01<00:00, 682.79it/s]

Epochs: 8 | Train Loss:  0.357 | Val Loss:  0.376 
Early stopping


In [6]:
from sklearn.metrics import f1_score
from tqdm import tqdm

outputs = []
labels = []

for inputs, b_labels in tqdm(val_dl):
    labels = [*labels, *b_labels]
    probabilities = model(inputs)
    classes = (probabilities >= 0.5).long()
    outputs = [*outputs, *classes.tolist()]

print('F1-Score: ',f1_score(outputs, labels))

100%|██████████| 1240/1240 [00:02<00:00, 540.44it/s]


F1-Score:  0.8256405172291699


In [5]:
acc = torch.tensor([1 if labels[i].item() == outputs[i] else 0 for i in range(len(outputs))])
acc.mean(dtype=float)

tensor(0.8448, dtype=torch.float64)